# MILK10k — Step 5b: melanoma-safe and tone-aware guarantees
Notebook 05 showed the group guarantee (≤ α of *all* treatable lesions reassured) holds, but hides much higher miss rates for **melanoma** and for the **darker skin-tone group**. This notebook compares four ways to calibrate the "actionable" side of the two-stage rule:

| variant | actionable side calibrated on | guarantee |
|---|---|---|
| `group` | all actionable lesions (notebook 05) | P(miss \| actionable) ≤ α |
| `worst_class` | each actionable class separately, use the **most conservative** quantile | P(miss \| class k) ≤ α for every treatable class k |
| `per_class_score` | 11-class Mondrian sets; actionable if **any** treatable class is in the set | P(miss \| class k) ≤ α for every k |
| `tone` | actionable lesions **within each tone group** (tone is known at intake) | P(miss \| actionable, tone g) ≤ α for every g |

MAL_OTH (9 lesions) is merged with MEL for calibration. Then: Pareto frontiers (miss vs benign reassurance) and bootstrap CIs by tone.

In [ ]:
import os, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
def find_file(name, tops=("/kaggle/input", "/kaggle/working")):
    for top in tops:
        for root, dirs, files in os.walk(top, followlinks=True):
            if name in files: return os.path.join(root, name)
    return None
NPZ = find_file("oof_probs_fusion.npz"); FOLDS_CSV = find_file("milk10k_lesions_folds.csv")
assert NPZ and FOLDS_CSV, "attach notebook-o3 and notebook-o2 outputs"
OUT = "/kaggle/working/stage5b"; os.makedirs(OUT, exist_ok=True)
les = pd.read_csv(FOLDS_CSV)
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
ACTIONABLE = ["AKIEC", "BCC", "MAL_OTH", "MEL", "SCCKA"]; is_act_cls = np.isin(CLASSES, ACTIONABLE)
y = les["y"].values.astype(int); fold = les["fold"].values; n = len(y); y_act = is_act_cls[y]
calib_group = np.array([("MEL" if CLASSES[k] in ("MEL", "MAL_OTH") else CLASSES[k]) for k in y])   # MAL_OTH merged into MEL
ACT_GROUPS = ["AKIEC", "BCC", "MEL", "SCCKA"]
tone = pd.cut(les["skin_tone_class"], [-1, 2, 3, 5], labels=["darker(0-2)", "medium(3)", "lighter(4-5)"]).astype(str).values
TONES = ["darker(0-2)", "medium(3)", "lighter(4-5)"]
npz = np.load(NPZ); eps = 1e-9
def temp(P, T):
    L = np.log(P + eps) / T; L -= L.max(1, keepdims=True); E = np.exp(L); return E / E.sum(1, keepdims=True)
def nll(P, yy): return -np.log(P[np.arange(len(yy)), yy] + eps).mean()
PROB = {}
for p in ["CM", "CDM"]:
    P = npz[f"late_drop|{p}"].astype(float); out = np.zeros_like(P)
    for f in range(5):
        tr, te = fold != f, fold == f
        T = minimize_scalar(lambda t: nll(temp(P[tr], t), y[tr]), bounds=(0.2, 5), method="bounded").x
        out[te] = temp(P[te], T)
    PROB[p] = out
s_act = {p: PROB[p][:, is_act_cls].sum(1) for p in PROB}
print("actionable lesions per tone:", {g: int((y_act & (tone == g)).sum()) for g in TONES})
print("melanoma (+MAL_OTH) per tone:", {g: int(((calib_group == "MEL") & (tone == g)).sum()) for g in TONES})

## Two-stage rule with four calibration variants

In [ ]:
def q_upper(v, a):
    v = np.sort(v); m = len(v); k = int(np.ceil((m + 1) * (1 - a)))
    return np.inf if (m == 0 or k > m) else v[k - 1]

def sets(p, cal_fn, aA, aB, variant):
    """Returns (actionable_in_set, benign_in_set) for all lesions, cross-fitted over folds."""
    s, P = s_act[p], PROB[p]; inA, inB = np.zeros(n, bool), np.zeros(n, bool)
    for f in range(5):
        te, cal = fold == f, cal_fn(f)
        inB[te] = s[te] <= q_upper(s[cal & ~y_act], aB)
        if variant == "group":
            inA[te] = (1 - s[te]) <= q_upper(1 - s[cal & y_act], aA)
        elif variant == "worst_class":
            q = max(q_upper(1 - s[cal & (calib_group == g)], aA) for g in ACT_GROUPS)
            inA[te] = (1 - s[te]) <= q
        elif variant == "per_class_score":
            hit = np.zeros(te.sum(), bool)
            for g in ACT_GROUPS:
                ks = [CLASSES.index("MEL"), CLASSES.index("MAL_OTH")] if g == "MEL" else [CLASSES.index(g)]
                pk = P[:, ks].sum(1)
                hit |= (1 - pk[te]) <= q_upper(1 - pk[cal & (calib_group == g)], aA)
            inA[te] = hit
        elif variant == "tone":
            for g in TONES:
                m = te & (tone == g)
                inA[m] = (1 - s[m]) <= q_upper(1 - s[cal & y_act & (tone == g)], aA)
    return inA, inB

def decide(inA, inB): return np.where(inA & ~inB, "refer", np.where(inB & ~inA, "reassure", "ambiguous"))

def run(alpha, aB, strategy, variant):
    if strategy == "all":
        d = decide(*sets("CDM", lambda f: fold != f, alpha, aB, variant)); return np.where(d == "ambiguous", "refer", d), np.ones(n, bool)
    aA = alpha / 2 if strategy == "adaptive" else alpha
    d1 = decide(*sets("CM", lambda f: fold != f, aA, aB, variant))
    if strategy == "none": return np.where(d1 == "ambiguous", "refer", d1), np.zeros(n, bool)
    req = d1 == "ambiguous"
    d2 = decide(*sets("CDM", lambda f: (fold != f) & req, aA, aB, variant))
    final = d1.copy(); final[req] = np.where(d2[req] == "ambiguous", "refer", d2[req]); return final, req

def summ(final, req, mask=None):
    m = np.ones(n, bool) if mask is None else mask
    re, act = (final == "reassure") & m, y_act & m
    out = {"dermoscopy_rate": req[m].mean(), "referral_rate": (final == "refer")[m].mean(),
           "act_missed": (re & act).sum() / max(act.sum(), 1),
           "benign_reassured": (re & ~y_act).sum() / max((~y_act & m).sum(), 1)}
    for g in ACT_GROUPS:
        gm = (calib_group == g) & m; out[f"{g}_missed"] = (re & gm).sum() / max(gm.sum(), 1)
    return out

## Main comparison at α = 0.10 (miss budget), α_B = 0.20

In [ ]:
rows = []
for variant in ["group", "worst_class", "per_class_score", "tone"]:
    for strat in ["none", "adaptive", "all"]:
        final, req = run(0.10, 0.20, strat, variant)
        rows.append(dict(variant=variant, strategy=strat, **summ(final, req),
                         **{f"missed_{g}": summ(final, req, tone == g)["act_missed"] for g in TONES}))
main = pd.DataFrame(rows).set_index(["variant", "strategy"]); main.to_csv(f"{OUT}/main_comparison.csv"); main.round(3)

## Pareto frontiers: miss rate vs benign lesions reassured (sweep α, α_B)

In [ ]:
grid = []
for variant in ["group", "worst_class", "per_class_score", "tone"]:
    for strat in ["none", "adaptive", "all"]:
        for alpha in [0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20]:
            for aB in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
                final, req = run(alpha, aB, strat, variant)
                d = summ(final, req); d.update(variant=variant, strategy=strat, alpha=alpha, alpha_B=aB,
                                              darker_missed=summ(final, req, tone == "darker(0-2)")["act_missed"])
                grid.append(d)
grid = pd.DataFrame(grid); grid.to_csv(f"{OUT}/pareto_grid.csv", index=False)

def pareto(d, x, yv):   # points not dominated: lower x (miss) and higher y (reassured)
    d = d.sort_values([x, yv], ascending=[True, False]); best, keep = -1, []
    for _, r in d.iterrows():
        if r[yv] > best: keep.append(r); best = r[yv]
    return pd.DataFrame(keep)

fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
for a, xcol, ttl in zip(ax, ["act_missed", "MEL_missed", "darker_missed"],
                        ["all treatable lesions", "melanoma", "treatable, darker tones (0-2)"]):
    for (variant, strat), d in grid.groupby(["variant", "strategy"]):
        if variant not in ("group", "worst_class") and strat != "adaptive": continue
        pf = pareto(d, xcol, "benign_reassured")
        ls = {"none": ":", "adaptive": "-", "all": "--"}[strat]
        a.plot(pf[xcol] * 100, pf["benign_reassured"] * 100, ls, marker=".", label=f"{variant} / {strat}")
    a.set_xlabel(f"% missed ({ttl})"); a.set_ylabel("% benign lesions reassured"); a.set_xlim(0, 30)
ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{OUT}/fig_pareto.png", dpi=200); plt.show()

## Operating points matched on overall miss rate ≈ 5% (closest grid point per variant × strategy)

In [ ]:
pick = (grid.assign(dist=(grid["act_missed"] - 0.05).abs())
          .sort_values(["dist", "benign_reassured"], ascending=[True, False])
          .groupby(["variant", "strategy"]).head(1).set_index(["variant", "strategy"]).drop(columns="dist"))
pick.to_csv(f"{OUT}/matched_5pct.csv"); pick.round(3)

## Bootstrap 95% CIs by tone for the recommended setting

In [ ]:
REC = dict(alpha=0.10, aB=0.20, strategy="adaptive")      # edit after looking at the frontier
def boot(variant, B=500, seed=0):
    final, req = run(REC["alpha"], REC["aB"], REC["strategy"], variant)
    rng = np.random.RandomState(seed); rows = []
    for _ in range(B):
        idx = rng.randint(0, n, n); r = {}
        for g in TONES + ["all"]:
            m = np.ones(len(idx), bool) if g == "all" else tone[idx] == g
            act = y_act[idx] & m; re = final[idx] == "reassure"
            r[g] = (re & act).sum() / max(act.sum(), 1)
        mel = (calib_group[idx] == "MEL"); r["MEL"] = ((final[idx] == "reassure") & mel).sum() / max(mel.sum(), 1)
        rows.append(r)
    b = pd.DataFrame(rows)
    return pd.DataFrame({"missed": [summ(final, req, (tone == g) if g in TONES else None)["act_missed"] if g != "MEL"
                                    else summ(final, req)["MEL_missed"] for g in b.columns],
                         "ci_low": b.quantile(0.025).values, "ci_high": b.quantile(0.975).values}, index=b.columns)
cis = pd.concat({v: boot(v) for v in ["group", "worst_class", "per_class_score", "tone"]})
cis.to_csv(f"{OUT}/bootstrap_miss_by_tone.csv"); cis.round(3)